# Task 3 · Augmentation
Random crop (shift of up to 4 px), rotation (up to 15°), zoom (up to 1.4×) and horizontal and vertical flips, drawn
anew for every image in every epoch, so each epoch sees 40,000 new images. The Task 1 CNN is retrained for 15 epochs
without augmentation, with all five transformations, and with all but the vertical flip.

In [ ]:
from hw3 import *
from sklearn.metrics import confusion_matrix

B = json.load(open(BEST))['cnn']
X, Y, Xte, Yte = cifar()
torch.manual_seed(SEED)
ops = {'original': (), 'crop 4 px': ('crop',), 'rotation 15°': ('rotation',), 'zoom 1.4×': ('zoom',),
       'h-flip': ('h-flip',), 'v-flip': ('v-flip',), 'all five, random': AUG['all']}
fig, axes = plt.subplots(4, len(ops), figsize=(7.2, 4.3))
for r, c in enumerate((0, 3, 7, 9)):
    i = int((Y == c).nonzero()[0])
    for ax, (t, op) in zip(axes[r], ops.items()):
        img = augment(X[i:i + 1], op, fixed=op != AUG['all'])[0]
        ax.imshow((img / 255).clamp(0, 1).permute(1, 2, 0))
        ax.set_title(t if r == 0 else '')
        ax.axis('off')
save(fig, 'transforms')

In [ ]:
E = 15
base = dict(model=partial(CNN, B['arch']), lr=B['lr'], epochs=E)
R = cv({'aug_none': base, 'aug_novflip': base | dict(aug='novflip'), 'aug_all': base | dict(aug='all')})
G = {'no augmentation': ('aug_none', COL['cnn']), 'all but v-flip': ('aug_novflip', COL['novflip']),
     'all five': ('aug_all', COL['all'])}
pd.DataFrame({k: {m: ms(last(R[n], m)) for m in ('train_acc', 'val_acc', 'test_acc')} for k, (n, _) in G.items()}).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), gridspec_kw=dict(width_ratios=[1, 1, .7]))
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[:2])
for x, (n, c) in enumerate(G.values()):
    a = last(R[n], 'test_acc') * 100
    axes[2].errorbar(x, a.mean(), a.std(ddof=1), fmt='o', capsize=4, color=c)
axes[2].set(xlim=(-.5, 2.5), ylabel='test accuracy (%)')
axes[2].set_xticks(range(3), ['none', 'no v-flip', 'all five'], rotation=30, ha='right')
save(fig, 'aug_curves')

In [ ]:
conf = {n: np.array([confusion_matrix(Yte, r['pred'], normalize='true') for r in R[n]]) * 100 for n, _ in G.values()}
name = {n: k for k, (n, _) in G.items()}
top = max(['aug_novflip', 'aug_all'], key=lambda n: last(R[n], 'val_acc').mean())
D = conf[top].mean(0) - conf['aug_none'].mean(0)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.9))
for ax, M, t, cmap, lim in zip(axes, (conf['aug_none'].mean(0), conf[top].mean(0), D),
                               ('no augmentation (%)', f'{name[top]} (%)', 'difference (pp)'),
                               ('Blues', 'Blues', 'RdBu'), (100, 100, np.abs(D).max())):
    ax.imshow(M, cmap=cmap, vmin=-lim if cmap == 'RdBu' else 0, vmax=lim)
    for (r, c), v in np.ndenumerate(M):
        if abs(v) >= .5:
            ax.text(c, r, f'{v:.0f}', ha='center', va='center', fontsize=4.5, color='white' if abs(v) > .6 * lim else INK)
    ax.set_xticks(range(10), CLASSES, rotation=90, fontsize=6)
    ax.set_yticks(range(10), CLASSES, fontsize=6)
    ax.tick_params(labelleft=ax is axes[0])
    ax.set(title=t, xlabel='predicted')
    ax.grid(False)
axes[0].set_ylabel('true')
save(fig, 'confusion')

In [ ]:
pc = {n: np.array([np.diag(m) for m in conf[n]]) for n in conf}
fig, ax = plt.subplots(figsize=(7.2, 2.1))
for q, (k, (n, c)) in enumerate(G.items()):
    ax.errorbar(np.arange(10) + .22 * (q - 1), pc[n].mean(0), pc[n].std(0, ddof=1), fmt='o', ms=3.5, capsize=2, color=c, label=k)
ax.set_xticks(range(10), CLASSES, rotation=30, ha='right')
ax.set_ylabel('test accuracy (%)')
fig.legend(loc='outside upper center', ncol=3)
save(fig, 'per_class')

In [ ]:
gain = pc[top].mean(0) - pc['aug_none'].mean(0)
tex('t3', augEpochs=E, topName=name[top], bestClass=CLASSES[gain.argmax()], bestGain=f'{gain.max():+.1f}',
    worstClass=CLASSES[gain.argmin()], worstGain=f'{gain.min():+.1f}', augDiff=pdiff(R[top], R['aug_none']),
    vflipDiff=pdiff(R['aug_all'], R['aug_novflip']),
    **{m + n.removeprefix('aug_').capitalize(): ms(v) for n in R for m, v in (
        ('test', last(R[n], 'test_acc')), ('val', last(R[n], 'val_acc')), ('gap', last(R[n], 'train_acc') - last(R[n], 'val_acc')))})